# 02 · Isolation Forest & Local Outlier Factor

## What this notebook covers
Two workhorses of tabular anomaly detection that are still the go-to methods for most production use cases. One-class SVM is covered in the context section but not implemented — IF and LOF dominate it on every practical benchmark.

## One-class SVM — why it lost
One-class SVM finds the minimum-volume hypersphere in kernel space enclosing the training data. It works in theory but has two fatal practical flaws:
1. **Scaling**: training is O(n²–n³), making it unusable on datasets with >10k samples
2. **Hyperparameter sensitivity**: `nu` (expected outlier fraction) and `gamma` must be tuned carefully — small changes produce wildly different boundaries

For the occasional low-dimensional, small-sample problem it is still reasonable. In practice, Isolation Forest replaced it.

## Isolation Forest
The key insight: anomalies are *rare and different*, so they are isolated near the root of a random partition tree. Normal points require more partitions to isolate. The anomaly score is the average path length across an ensemble of isolation trees — short path = anomalous.

**Why it works well in production:**
- Linear O(n) training and inference
- No distance or density computation
- Insensitive to dimensionality (up to ~50 features)
- The `contamination` parameter is the only meaningful hyperparameter

## Local Outlier Factor
LOF compares the local density of a point to its neighbours' density. A point in a sparse region surrounded by dense regions is anomalous. LOF is excellent at finding **contextual** outliers that a global method like IF would miss — a point that is only anomalous relative to its local neighbourhood.

**Limitation**: LOF is a transductive method — it cannot score new points without re-fitting (sklearn's `novelty=True` mode approximates this but with caveats).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.datasets import make_blobs
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# Dataset: two normal clusters + outlier cloud
X_normal, _ = make_blobs(n_samples=400, centers=[[-5,-5],[5,5]], cluster_std=0.8, random_state=0)
X_outliers   = np.random.uniform(-12, 12, (20, 2))
X = np.vstack([X_normal, X_outliers])
y_true = np.array([0]*400 + [1]*20)

print(f"Dataset: {len(X)} points  |  {y_true.sum()} true anomalies ({y_true.mean():.1%})")

In [ ]:
# ── Isolation Forest ───────────────────────────────────────────────────────────
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=0)
iso.fit(X)
scores_if = -iso.score_samples(X)   # higher = more anomalous
preds_if  = iso.predict(X) == -1   # -1 = anomaly

auc_if = roc_auc_score(y_true, scores_if)
ap_if  = average_precision_score(y_true, scores_if)
print(f"Isolation Forest: ROC-AUC={auc_if:.4f}  AP={ap_if:.4f}")
print(f"  Flagged: {preds_if.sum()} points")

In [ ]:
# ── Local Outlier Factor ───────────────────────────────────────────────────────
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=False)
lof.fit(X)
scores_lof = -lof.negative_outlier_factor_  # higher = more anomalous
preds_lof  = lof.fit_predict(X) == -1

auc_lof = roc_auc_score(y_true, scores_lof)
ap_lof  = average_precision_score(y_true, scores_lof)
print(f"Local Outlier Factor: ROC-AUC={auc_lof:.4f}  AP={ap_lof:.4f}")
print(f"  Flagged: {preds_lof.sum()} points")

In [ ]:
# Visualise decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
xx, yy = np.meshgrid(np.linspace(-14, 14, 200), np.linspace(-14, 14, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

for ax, model, scores, title in zip(
    axes,
    [iso, lof],
    [scores_if, scores_lof],
    ["Isolation Forest", "Local Outlier Factor"]
):
    if hasattr(model, "score_samples"):
        Z = -model.score_samples(grid).reshape(xx.shape)
    else:
        # LOF in novelty=False mode can't score new points — use IF scores for viz
        Z = -iso.score_samples(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=20, cmap="RdYlGn_r", alpha=0.6)
    ax.scatter(X[y_true==0, 0], X[y_true==0, 1], c="steelblue", s=12, alpha=0.7, label="Normal")
    ax.scatter(X[y_true==1, 0], X[y_true==1, 1], c="red", s=60, marker="x", label="True anomaly", zorder=5)
    ax.set_title(f"{title}  (AUC={roc_auc_score(y_true, scores):.3f})")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{base}/02_iforest_lof.png", dpi=120)
plt.show()

In [ ]:
# Contamination sensitivity analysis
contaminations = [0.01, 0.02, 0.05, 0.10, 0.15]
rows = []
for c in contaminations:
    iso_c = IsolationForest(n_estimators=200, contamination=c, random_state=0)
    iso_c.fit(X)
    p = iso_c.predict(X) == -1
    prec = (p & y_true.astype(bool)).sum() / p.sum() if p.sum() > 0 else 0
    rec  = (p & y_true.astype(bool)).sum() / y_true.sum()
    rows.append({"contamination": c, "flagged": p.sum(), "precision": round(prec,3), "recall": round(rec,3)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nKey insight: AUC is contamination-independent; precision/recall are not.")
print(f"AUC is always: {auc_if:.4f} regardless of contamination setting.")